# GameTheory-21 : Capstone — la sous-série SocialChoice en une heure

**Navigation** : [<< 17-MultiAgent-RL](GameTheory-17-MultiAgent-RL-Python.ipynb) | [Index](README.md) | [Sous-série SocialChoice >>](SocialChoice/README.md)

**Side tracks** : ce carnet EST l'escalier vers la sous-série [SocialChoice](SocialChoice/README.md) — il la présente et n'en remplace aucun carnet.

Ce carnet est le **capstone** qui referme le parcours principal de la série GameTheory et ouvre sa sous-série SocialChoice, consacrée à l'agrégation des préférences. Il répond à trois questions, dans l'ordre où un lecteur du parcours les pose :

1. **Ce que la sous-série démontre** — les résultats qu'elle établit, et pourquoi ils comptent au-delà de la théorie des jeux ;
2. **Ce qu'elle suppose** — les prérequis, palier par palier, pour y entrer au bon endroit ;
3. **Par où y entrer** — trois parcours de lecture selon votre niveau.

Il le fait en **trois expériences de dix minutes** (une simulation du théorème d'Arrow, un cycle de Condorcet, un mariage stable de Gale-Shapley) et en **citant le lake Lean** de la sous-série, dont les théorèmes sont prouvés formellement — pas seulement simulés.

***

## Ce que ce notebook suppose

- **Prérequis** : le parcours [GameTheory 01-16](README.md), en particulier la [forme normale](GameTheory-02-NormalForm-Python.ipynb) (préférences ordonnées, notations) et le [palier 16](GameTheory-16-MechanismDesign-Python.ipynb) (le principe de révélation et l'appariement stable, dont Gale-Shapley est le moteur). Aucun prérequis en théorie du choix social : tout est introduit ici.
- **Outil** : Python 3, bibliothèque standard uniquement. Le carnet s'exécute de bout en bout, exercices non complétés compris.
- **Public** : **Licence**. Les extensions vers la recherche (formalisation Lean, encodage SAT, dividendes de Harsanyi) sont citées et liées, pas supposées.


## 1. Ce que la sous-série démontre

La théorie du choix social étudie une question qui paraît simple : **comment transformer des préférences individuelles en une décision collective ?** Un vote, un classement, un comité, un appariement — dans chaque cas, une règle agrège des ordres individuels en un ordre (ou un choix) social.

La réponse de la sous-série tient en une famille de **théorèmes d'impossibilité** et en trois **changements d'objet** :

| Angle | Carnet | Ce qu'il établit | Durée | Public |
|---|---|---|---|---|
| Simulation d'Arrow | [SC-01](SocialChoice/01-Arrow-Impossibility-Theorem.ipynb) | Aucune règle ne satisfait simultanément Pareto, IIA et non-dictature (preuve de Geanakoplos pas à pas, testée par le code) | 45 min | Licence |
| Preuve formelle | [SC-02](SocialChoice/01b-Lean-SocialChoice-Formal.ipynb) | Arrow, Sen et l'électeur médian **prouvés en Lean 4**, 0 `sorry` | 80 min | Recherche |
| Méthodes de vote | [SC-03](SocialChoice/03-Voting-Methods.ipynb) | Condorcet, Borda, Copeland, Downs : chaque règle a son paradoxe | 35 min | Découverte |
| Vérification SAT | [SC-04](SocialChoice/04-Computational-Aggregation-SAT-Z3.ipynb) | L'impossibilité devient un `UNSAT` : la machine vérifie tous les cas finis | 45 min | Licence |
| Manipulation | [SC-05](SocialChoice/05-Gibbard-Satterthwaite.ipynb) | Gibbard-Satterthwaite : toute règle non triviale est manipulable — le témoin s'exhibe par le code | 30 min | Licence |
| Agrégation coopérative | [SC-06](SocialChoice/06-Mobius-Aggregation-Pouvoir-Manipulation.ipynb) | Möbius sur le treillis des coalitions : dividendes de Harsanyi, poids ≠ pouvoir | 40 min | Recherche |
| Comités | [SC-07](SocialChoice/07-Committees-Core.ipynb) | Élire un comité : le core existe toujours (quotas Hare/Droop, certificats) | 40 min | Recherche |
| Mariage stable | [SC-08](SocialChoice/02-Stable-Marriage-Gale-Shapley.ipynb) | Gale-Shapley : couplage parfait, stabilité, man-optimalité — jumeau d'un lake Lean 0 `sorry` | 35 min | Licence |

Les trois impossibilités fondatrices (Arrow 1951, Sen 1970, Gibbard-Satterthwaite 1973) disent la même chose trois fois, pour trois objets : **un ordre social, une liberté individuelle, une règle de vote** — sous des axiomes « raisonnables », quelque chose doit casser. Les trois changements d'objet (SC-06 : agréger des valeurs de coalition ; SC-07 : élire un comité ; SC-08 : apparier deux populations) montent ce qui reste possible quand on change la question.

Les trois expériences qui suivent mettent chaque fois **un seul résultat en main**, dans le formalisme minimal pour le voir.


## 2. Trois expériences de dix minutes


### 2.1 Le théorème d'Arrow par la simulation

Trois axiomes, trois règles, un verdict. Les axiomes :

- **Pareto (faible)** : si *tous* les électeurs préfèrent `x` à `y`, la règle doit classer `x` avant `y` ;
- **IIA** (indépendance des alternatives non pertinentes) : le classement social de `x` vs `y` ne doit dépendre que des classements individuels de `x` vs `y` — pas des positions d'une troisième alternative `z` ;
- **Non-dictature** : aucun électeur seul ne impose son classement.

Le théorème d'Arrow (1951) : avec **3 alternatives ou plus**, aucune fonction de bien-être social ne satisfait les trois. La cellule suivante définit le matériel (profils et règles), la suivante teste les axiomes.


In [1]:
# Materiau commun aux experiences : un profil = liste d'ordres individuels.
# Convention : chaque ordre est une liste des alternatives, de la meilleure a la pire.

def borda(profil, alts):
    """Score de Borda : chaque electeur donne m-1, m-2, ..., 0 points a ses alternatives."""
    m = len(alts)
    scores = {a: 0 for a in alts}
    for ordre in profil:
        for rang, alt in enumerate(ordre):
            scores[alt] += m - 1 - rang
    return scores

def pluralite(profil, alts):
    """Score de pluralite : 1 point a l'alternative classee premiere par chaque electeur."""
    scores = {a: 0 for a in alts}
    for ordre in profil:
        scores[ordre[0]] += 1
    return scores

def dictateur(k, profil, alts):
    """Score du dictateur k : son ordre personnel devient l'ordre social."""
    m = len(alts)
    return {alt: m - 1 - rang for rang, alt in enumerate(profil[k])}

def ordre_social(scores):
    """Ordre social decroissant selon les scores (lesegalites restent visibles)."""
    return sorted(scores, key=scores.get, reverse=True)

ALTS = ["A", "B", "C"]
print("Regles definies : borda, pluralite, dictateur.")


Regles definies : borda, pluralite, dictateur.


In [2]:
# Test des trois axiomes sur les trois regles.
# Pareto : pour toute paire unanime x > y, l'ordre social doit placer x avant y.
# IIA : deux profils P1/P2 ou CHAQUE electeur garde le meme ordre entre A et B,
#       mais ou l'ordre social A vs B bascule -> violation.
# Non-dictature : la regle ne doit pas coincider systematiquement avec l'ordre d'un electeur.

def pareto_ok(score_fn, profil, alts):
    ordre = ordre_social(score_fn(profil, alts) if callable(score_fn) else score_fn)
    pos = {a: i for i, a in enumerate(ordre)}
    for i, x in enumerate(alts):
        for y in alts[i+1:]:
            if all(o.index(x) < o.index(y) for o in profil) and not pos[x] < pos[y]:
                return False
    return True

# Temoins IIA (mesures ci-dessous) :
#  pluralite : P1 -> social A>B ; P2 -> social B>A (preferences A-vs-B individuelles identiques)
#  borda     : P1' -> social B>A ; P2' -> social A>B (idem, 5 electeurs)
P1_PLU = [["A", "B", "C"], ["B", "C", "A"], ["A", "B", "C"]]
P2_PLU = [["C", "A", "B"], ["B", "C", "A"], ["C", "A", "B"]]
P1_BOR = [["A", "B", "C"], ["A", "B", "C"], ["A", "B", "C"], ["B", "C", "A"], ["B", "C", "A"]]
P2_BOR = [["A", "C", "B"], ["A", "C", "B"], ["A", "C", "B"], ["B", "A", "C"], ["B", "A", "C"]]

def iia_temoin(fn, P1, P2, alts, x="A", y="B"):
    """True si le temoin ne bascule PAS (IIA respectee sur ce cas), False s'il bascule."""
    def soc(P):
        s = fn(P, alts)
        return s[x] > s[y]
    return soc(P1) == soc(P2) and (all(o.index(x) < o.index(y) for o in P1) ==
                                   all(o.index(x) < o.index(y) for o in P2))

PROFIL_ARROW = [["A", "B", "C"], ["B", "A", "C"], ["A", "C", "B"]]

lignes = []
for nom, fn, iia in [
    ("Borda", lambda P, a: borda(P, a), iia_temoin(borda, P1_BOR, P2_BOR, ALTS)),
    ("Pluralite", lambda P, a: pluralite(P, a), iia_temoin(pluralite, P1_PLU, P2_PLU, ALTS)),
]:
    lignes.append((nom, pareto_ok(fn, PROFIL_ARROW, ALTS), iia, True))
# Dictateur 0 : Pareto et IIA tiennent par construction, la non-dictature tombe.
lignes.append(("Dictateur(v1)", pareto_ok(lambda P, a: dictateur(0, P, a), PROFIL_ARROW, ALTS), True, False))

print(f"{'Regle':<14} {'Pareto':<8} {'IIA':<6} {'Non-dictature':<14} Verdict")
for nom, p, i, nd in lignes:
    verts = sum([p, i, nd])
    print(f"{nom:<14} {'OK' if p else 'VIOLE':<8} {'OK' if i else 'VIOLE':<6} "
          f"{'OK' if nd else 'VIOLE':<14} {verts}/3 "
          + ("-> impossible" if verts < 3 else ""))
print()
print("Details des temoins IIA :")
print("  Pluralite : P1 scores", pluralite(P1_PLU, ALTS), "-> ordre", ordre_social(pluralite(P1_PLU, ALTS)))
print("              P2 scores", pluralite(P2_PLU, ALTS), "-> ordre", ordre_social(pluralite(P2_PLU, ALTS)))
print("  Borda     : P1' scores", borda(P1_BOR, ALTS), "-> ordre", ordre_social(borda(P1_BOR, ALTS)))
print("              P2' scores", borda(P2_BOR, ALTS), "-> ordre", ordre_social(borda(P2_BOR, ALTS)))


Regle          Pareto   IIA    Non-dictature  Verdict
Borda          OK       VIOLE  OK             2/3 -> impossible
Pluralite      OK       VIOLE  OK             2/3 -> impossible
Dictateur(v1)  OK       OK     VIOLE          2/3 -> impossible

Details des temoins IIA :
  Pluralite : P1 scores {'A': 2, 'B': 1, 'C': 0} -> ordre ['A', 'B', 'C']
              P2 scores {'A': 0, 'B': 1, 'C': 2} -> ordre ['C', 'B', 'A']
  Borda     : P1' scores {'A': 6, 'B': 7, 'C': 2} -> ordre ['B', 'A', 'C']
              P2' scores {'A': 8, 'B': 4, 'C': 3} -> ordre ['A', 'B', 'C']


**Lecture du résultat.** Aucune ligne n'est verte des trois colonnes : Borda et la pluralité violent l'IIA (déplacer **C** — une alternative perdante — suffit à inverser le classement social de A vs B), la dictature viole la non-dictature. C'est le théorème d'Arrow **vu par le code** : chaque axiome isolé est satisfaisable, la conjonction ne l'est pas — sur ces exemples.

Ce que la simulation **ne montre pas** : elle teste trois règles sur un profil. Arrow dit que **toutes** les règles (avec 3+ alternatives) échouent — une infinité de cas que seul un raisonnement couvre. SC-01 déroule la preuve de Geanakoplos pas à pas ; SC-04 la fait vérifier par un solveur SAT sur les profils à 3 électeurs ; le lake Lean (section 3 ci-dessous) la prouve une fois pour toutes.


### 2.2 Le cycle de Condorcet

Le marquis de Condorcet a proposé en 1785 un critère qui semble inattaquable : si une alternative bat toutes les autres en duel majoritaire, c'est elle qu'il faut élire. Le profil ci-dessous — le plus petit profil paradoxaux — montre pourquoi ce critère peut être **vide** : personne ne bat tout le monde.


In [3]:
# Le profil cyclique minimal : 3 electeurs, 3 alternatives.
CYCLE = [["A", "B", "C"], ["B", "C", "A"], ["C", "A", "B"]]

def duels(profil, alts):
    """Pour chaque paire, qui gagne au vote majoritaire (et de combien) ?"""
    for i, x in enumerate(alts):
        for y in alts[i+1:]:
            pour_x = sum(1 for o in profil if o.index(x) < o.index(y))
            pour_y = len(profil) - pour_x
            vainqueur = x if pour_x > pour_y else (y if pour_y > pour_x else "egalite")
            print(f"  {x} vs {y} : {pour_x}-{pour_y} -> {vainqueur}")

print("Duels majoritaires sur le profil cyclique :")
duels(CYCLE, ALTS)
print()
print("Scores de Borda sur le meme profil :", borda(CYCLE, ALTS))
print("Scores de pluralite :", pluralite(CYCLE, ALTS))
print("Scores de Copeland (1 victoire, 0.5 nul, 0 defaite) : a vous -- Exercice 1.")


Duels majoritaires sur le profil cyclique :
  A vs B : 2-1 -> A
  A vs C : 1-2 -> C
  B vs C : 2-1 -> B

Scores de Borda sur le meme profil : {'A': 3, 'B': 3, 'C': 3}
Scores de pluralite : {'A': 1, 'B': 1, 'C': 1}
Scores de Copeland (1 victoire, 0.5 nul, 0 defaite) : a vous -- Exercice 1.


**Lecture du résultat.** A bat B (2-1), B bat C (2-1), C bat A (2-1) : le duel majoritaire **cycle**, il n'existe pas de gagnant de Condorcet. Pire : Borda et la pluralité rendent toutes deux un **égalité parfaite** (3 points chacune) — les deux règles sont muettes exactement là où le critère de Condorcet est vide.

Ce n'est pas un accident de fabrication : le théorème de birds-of-a-feather n'existe pas, mais le **paradoxe d'Arrow** (ci-dessus) et ce cycle sont les deux faces d'une même rigidité — agréger des ordres *sans jamais perdre d'information* est impossible, chaque règle choisit quelle information elle sacrifie. SC-03 déroule le bestiaire complet (Condorcet, Borda, Copeland, Dowd, paradoxe de l'échange, électeur médian) et montre quel paradoxe chaque règle accepte.


### 2.3 Mariage stable : Gale-Shapley

Troisième changement d'objet : on n'élit plus, on **apparie**. Deux populations (par convention des « proposants » et des « disposantes ») expriment chacune un ordre de préférence sur l'autre. Un appariement est **stable** si aucune paire (un proposant, une disposante) ne préfère mutuellement quitter leurs partenaires respectifs pour se mettre ensemble — une telle paire est dite **bloquante**.

L'algorithme de Gale-Shapley (1962) : les proposants demandent, les disposantes gardent provisoirement le meilleur et refusent le reste, jusqu'à ce que plus personne ne demande. Il produit toujours un appariement stable — et il est **optimal pour les proposants** : aucun proposant ne peut faire mieux dans aucun appariement stable.


In [4]:
# Gale-Shapley sur 4x4 preferences fixes (deterministe).
PROPOSANTS = {
    "h1": ["f1", "f2", "f3", "f4"],
    "h2": ["f2", "f1", "f4", "f3"],
    "h3": ["f3", "f4", "f1", "f2"],
    "h4": ["f4", "f3", "f2", "f1"],
}
DISPOSANTES = {
    "f1": ["h2", "h1", "h4", "h3"],
    "f2": ["h1", "h2", "h3", "h4"],
    "f3": ["h4", "h3", "h2", "h1"],
    "f4": ["h3", "h4", "h1", "h2"],
}

def gale_shapley(proposants, disposantes):
    """Deferred acceptance : les proposants demandent, les disposantes choisissent."""
    libres = list(proposants)
    prochain_choix = {p: 0 for p in proposants}
    fiance = {d: None for d in disposantes}
    demandes = 0
    while libres:
        p = libres.pop(0)
        d = proposants[p][prochain_choix[p]]
        prochain_choix[p] += 1
        demandes += 1
        if fiance[d] is None:
            fiance[d] = p
        elif disposantes[d].index(p) < disposantes[d].index(fiance[d]):
            libres.append(fiance[d])
            fiance[d] = p
        else:
            libres.append(p)
    return {p: d for d, p in fiance.items()}, demandes

M_hommes, nb_demandes = gale_shapley(PROPOSANTS, DISPOSANTES)
M_femmes_brut, _ = gale_shapley(DISPOSANTES, PROPOSANTS)  # {disposante: proposant}
M_femmes = {p: d for d, p in M_femmes_brut.items()}        # rebase en {proposant: disposante}
print("Appariement optimal pour les proposants :", dict(sorted(M_hommes.items())),
      f"({nb_demandes} demandes)")
print("Appariement optimal pour les disposantes :", dict(sorted(M_femmes.items())))


Appariement optimal pour les proposants : {'h1': 'f1', 'h2': 'f2', 'h3': 'f3', 'h4': 'f4'} (4 demandes)
Appariement optimal pour les disposantes : {'h1': 'f2', 'h2': 'f1', 'h3': 'f4', 'h4': 'f3'}


In [5]:
# Verification des deux promesses de Gale-Shapley, sur les deux appariements.

def paire_bloquante(M, proposants, disposantes):
    """Cherche une paire (p, d) qui prefererait mutuellement se quitter."""
    inverse = {d: p for p, d in M.items()}
    for p in proposants:
        for d in disposantes:
            if d == M[p]:
                continue
            actuel_p, actuel_d = M[p], inverse[d]
            if (proposants[p].index(d) < proposants[p].index(actuel_p)
                    and disposantes[d].index(p) < disposantes[d].index(actuel_d)):
                return (p, d)
    return None

def rang(pref, x):
    return pref.index(x) + 1  # rang 1 = meilleur

for nom, M in [("optimal proposants", M_hommes), ("optimal disposantes", M_femmes)]:
    bloq = paire_bloquante(M, PROPOSANTS, DISPOSANTES)
    print(f"{nom:<20} stable : {'OUI' if bloq is None else 'NON, paire bloquante ' + str(bloq)}")

print()
print("Rangs obtenus par les proposants (1 = meilleur) :")
print(f"{'':<18} {'h1':>4} {'h2':>4} {'h3':>4} {'h4':>4}")
r_h = [rang(PROPOSANTS[h], M_hommes[h]) for h in ["h1", "h2", "h3", "h4"]]
r_f = [rang(PROPOSANTS[h], M_femmes[h]) for h in ["h1", "h2", "h3", "h4"]]
print(f"{'opt. proposants':<18} {r_h[0]:>4} {r_h[1]:>4} {r_h[2]:>4} {r_h[3]:>4}")
print(f"{'opt. disposantes':<18} {r_f[0]:>4} {r_f[1]:>4} {r_f[2]:>4} {r_f[3]:>4}")
print()
print("Chaque proposant est au moins aussi bien place dans l'appariement "
      f"qu'il initie : {all(a <= b for a, b in zip(r_h, r_f))}")


optimal proposants   stable : OUI
optimal disposantes  stable : OUI

Rangs obtenus par les proposants (1 = meilleur) :
                     h1   h2   h3   h4
opt. proposants       1    1    1    1
opt. disposantes      2    2    2    2

Chaque proposant est au moins aussi bien place dans l'appariement qu'il initie : True


**Lecture du résultat.** Les deux appariements sont **stables** (aucune paire bloquante — la vérification tourne sur toutes les pires paires possibles), mais ils **diffèrent** : dans celui que les proposants initient, chaque proposant est au rang qu'il a dans l'autre ou mieux. La stabilité n'est pas neutre — **celui qui demande est favorisé**. C'est exactement pourquoi le mécanisme d'admission aux internats américains a changé de côté en 1998 : laisser les étudiants proposer (et non les hôpitaux) leur donnait l'optimalité sans perdre la stabilité.

SC-08 pousse chaque affirmation jusqu'à la preuve : couplage parfait garanti, man-optimalité **vérifiée par énumération exhaustive** de tous les appariements stables, et — côté formel — le lake Lean `game_theory_lean/StableMarriage/` prouve la correction de l'algorithme sans un seul `sorry`.


## 3. Ce que le lake Lean prouve (et que la simulation ne prouve pas)

La sous-série porte un **lake Lean 4** de référence : le dossier `game_theory_lean/SocialChoice/`, construit sur les notes de Dominik Peters. Les cellules suivantes le lisent **dans le dépôt** — ce qu'elles affichent est mesuré, pas recopié d'une documentation.


In [6]:
# Lecture directe du lake : enonce d'Arrow tel que prouve, et inventaire mesure.
import pathlib, re

LAKE = pathlib.Path("game_theory_lean") / "SocialChoice"
fleche = (LAKE / "Arrow.lean").read_text(encoding="utf-8")

# Extraire la docstring du theoreme arrow et sa signature (par index : un motif
# /--.*-/ non ancre avalerait toutes les defs intermediaires du fichier).
i = fleche.index("theorem arrow")
j = fleche.rfind("/--", 0, i)  # docstring immediatement precedente
k = fleche.index("-/", j)
docstring = fleche[j + 3 : k].strip()
signature = fleche[i : fleche.index(" := ", i)]
print("Enonce tel qu'il vit dans Arrow.lean :")
print("  " + "\n".join(l.strip() for l in docstring.splitlines()))
print()
print("Signature Lean :")
for l in signature.splitlines():
    print("  " + l)


Enonce tel qu'il vit dans Arrow.lean :
  Théorème d'impossibilité d'Arrow :
Toute fonction de bien-être social sur au moins 3 alternatives qui satisfait
le Pareto faible et l'indépendance des alternatives non pertinentes doit
être une dictature.

Signature Lean :
  theorem arrow (f : SWF ι σ) (X : Finset σ)
      (hwp : weak_pareto f X) (hind : ind_of_irr_alts f X) (hX : 3 ≤ X.card) :
      is_dictatorship f X


In [7]:
# Inventaire mesure du lake : modules, theoremes/lemmes, sorry.
modules = sorted(p.name for p in LAKE.glob("*.lean")
                 if not p.name.endswith("_en.lean") and not p.name.startswith("_"))
total_thm, total_sorry = 0, 0
for mod in modules:
    texte = (LAKE / mod).read_text(encoding="utf-8")
    thm = len(re.findall(r"^(?:theorem|lemma) ", texte, re.M))
    sorry = texte.count("sorry")
    total_thm += thm
    total_sorry += sorry
print(f"{len(modules)} modules (plus leurs miroirs _en) : "
      + ", ".join(m.replace(".lean", "") for m in modules))
print(f"{total_thm} theoremes/lemmes, {total_sorry} 'sorry'.")
print("La CI (.github/workflows/lean-social-choice.yml) certifie ce zero-sorry a chaque push.")


8 modules (plus leurs miroirs _en) : AMD, Arrow, Basic, Framework, MechanismDesign, Sen, SortedListCounting, Voting
109 theoremes/lemmes, 0 'sorry'.
La CI (.github/workflows/lean-social-choice.yml) certifie ce zero-sorry a chaque push.


**Lecture du résultat.** L'énoncé que la simulation a *effleuré* sur un profil, le lake le prouve **pour toute fonction de bien-être social sur au moins 3 alternatives** — l'infinité de cas que ni la simulation (elle échantillonne), ni SAT (il vérifie les cas finis, SC-04) ne couvrent. C'est la gradation complète de la sous-série : **voir** (simuler, SC-01), **vérifier** (SAT, SC-04), **prouver** (Lean, SC-02). Le capstone s'arrête à la première marche ; les deux autres sont le parcours Recherche ci-dessous.


## 4. Par où y entrer

Trois parcours de lecture dans la sous-série, selon ce que vous cherchez :

- **Découverte** (~1 h) : [SC-03 Méthodes de vote](SocialChoice/03-Voting-Methods.ipynb) puis [SC-05 Gibbard-Satterthwaite](SocialChoice/05-Gibbard-Satterthwaite.ipynb). Aucun prérequis au-delà de ce capstone : les paradoxes électoraux d'abord, la manipulation ensuite.
- **Licence** (~2 h) : [SC-01 Arrow par la simulation](SocialChoice/01-Arrow-Impossibility-Theorem.ipynb) (la preuve de Geanakoplos déroulée par le code), puis [SC-04 SAT/Z3](SocialChoice/04-Computational-Aggregation-SAT-Z3.ipynb) (l'impossibilité devenue `UNSAT`), puis [SC-08 Gale-Shapley](SocialChoice/02-Stable-Marriage-Gale-Shapley.ipynb) (la face constructive).
- **Recherche** (~4 h) : [SC-02 Choix social formel en Lean](SocialChoice/01b-Lean-SocialChoice-Formal.ipynb) (Arrow, Sen, électeur médian, tour du lake de D. Peters), [SC-06 Möbius et pouvoir](SocialChoice/06-Mobius-Aggregation-Pouvoir-Manipulation.ipynb), [SC-07 Comités et core](SocialChoice/07-Committees-Core.ipynb).

Le [README de la sous-série](SocialChoice/README.md) porte le parcours complet, les durées, et le diagramme des dépendances entre les huit angles.


## 5. Exercices

Trois exercices pour passer de lecteur à praticien. Chaque stub s'exécute (il rend `None` ou affiche un message) — complétez-le, la cellule de vérification qui suit reste valable.


In [8]:
# Exercice 1 -- Copeland sur le profil cyclique.
# Score de Copeland : +1 par victoire en duel majoritaire, +0.5 par egalite, 0 par defaite.
# Indice : reutilisez la logique de `duels` ci-dessus ; le profil CYCLE est defini en 2.2.
# Etape 1 : pour chaque alternative, compter victoires et egalites.
# Etape 2 : retourner le dict {alternative: score}.

def copeland(profil, alts):
    # TODO etudiant
    return None

print("Exercice a completer : copeland(CYCLE, ALTS) doit rendre un dict -- "
      "que remarquez-vous sur le profil cyclique ?")


Exercice a completer : copeland(CYCLE, ALTS) doit rendre un dict -- que remarquez-vous sur le profil cyclique ?


In [9]:
# Exercice 2 -- Trouver une manipulation de Borda.
# Avec le profil PROFIL_ARROW (2.1) et l'electeur 2 (indice 1), cherchez un bulletin
# NON sincere (un ordre different de ses preferences reelles) qui ameliore son issue,
# les autres bulletins restant fixes.
# Indice : enumeratez les 6 ordres possibles pour l'electeur 2 ; comparez le gagnant.
# Etape 1 : gagnant sincere = argmax borda(PROFIL_ARROW).
# Etape 2 : pour chaque ordre alternatif, gagnant hypothetique ; garder la premiere amelioration
#           pour l'electeur 2 (l'ordre reel de l'electeur 2 est PROFIL_ARROW[1]).

def manipule_borda(profil, alts, indice_electeur):
    # TODO etudiant
    return None

print("Exercice a completer : manipule_borda(PROFIL_ARROW, ALTS, 1) doit rendre "
      "un ordre non sincere qui ameliore l'electeur 2, ou None s'il n'en existe pas.")


Exercice a completer : manipule_borda(PROFIL_ARROW, ALTS, 1) doit rendre un ordre non sincere qui ameliore l'electeur 2, ou None s'il n'en existe pas.


In [10]:
# Exercice 3 -- Une paire bloquante a la main.
# On retire h1 de l'appariement optimal-proposants et on le fiance a f4 par decree.
# Cette paire (h1, f4) est-elle bloquante ? Et lesquels des deux ex-fiances veulent fuir ?
# Indice : comparez, pour h1 ET pour f4, le rang du partenaire impose a celui qu'ils avaient.
# Etape 1 : appariement modifie M2 = copie de M_hommes avec la permutation.
# Etape 2 : appliquez paire_bloquante(M2, ...) et interpretez la paire rendue.

def appariement_modifie():
    # TODO etudiant
    return None

print("Exercice a completer : construisez M2 et rendez la paire bloquante rendue par "
      "paire_bloquante(M2, PROPOSANTS, DISPOSANTES).")


Exercice a completer : construisez M2 et rendez la paire bloquante rendue par paire_bloquante(M2, PROPOSANTS, DISPOSANTES).


## Conclusion

Ce capstone a refermé une boucle : le parcours GameTheory vous a appris à **chercher un équilibre** (que feront des joueurs rationnels ?) ; la sous-série SocialChoice demande la question duale — **comment décider ensemble** quand les préférences divergent ? — et y répond par des théorèmes d'impossibilité dressés par des preuves, des solveurs et des simulations.

- Pour suivre le fil complet : [README de la sous-série SocialChoice](SocialChoice/README.md) ;
- Pour revenir au parcours : le [palier 16](GameTheory-16-MechanismDesign-Python.ipynb) (mécanismes et révélation) est le point d'ancrage naturel ; les [extensions au-delà du parcours](README.md#extensions-au-delà-du-parcours) poursuivent dans une autre direction ;
- Le lake formel : `game_theory_lean/SocialChoice/` (certifié sans `sorry` par la CI), visité en détail par [SC-02](SocialChoice/01b-Lean-SocialChoice-Formal.ipynb).

**Ce que ce notebook suppose** était en tête de carnet ; ce qu'il laisse ouvert est dans les exercices — le capstone se referme sur eux.
